# **Proyek ETL Pipeline & Analisis Pemasaran Perbankan (Bank Marketing)**
## **Layer 1: Exploratory Data Analysis (EDA) & Bronze Layer Ingestion**

Notebook ini merupakan tahap pertama dari arsitektur data lakehouse Medallion. Pada tahap ini dilakukan:
1. **Pemuatan Data Mentah (Raw Ingestion):** Mengunduh dataset Bank Marketing dari UCI Machine Learning Repository.
2. **Exploratory Data Analysis (EDA):** Analisis kualitas data (skema, statistik, missing values, duplikasi) dan agregasi eksploratif menggunakan Spark SQL.
3. **Ingestion ke Bronze Layer:** Menyimpan data mentah ke dalam format Parquet dengan penambahan metadata penelusuran (`ingested_at`, `source_file`) serta ekspor ke CSV tunggal.

### **Informasi Sumber Data**

* **Dataset:** Bank Marketing
* **Sumber:** [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/222/bank+marketing)
* **Format:** CSV
* **Periode/Target:** Kampanye pemasaran langsung (telepon) dari lembaga perbankan Portugis untuk memprediksi apakah klien akan berlangganan deposito berjangka (`y`).
* **Fitur Utama:** `age`, `job`, `marital`, `education`, `default`, `balance`, `housing`, `loan`, `contact`, `day`, `month`, `duration`, `campaign`, `pdays`, `previous`, `poutcome`, `y`
* **Lisensi:** Creative Commons Attribution 4.0 International (CC BY 4.0)
* **Metode:** Studi kasus kampanye pemasaran telepon langsung.

### **Pertanyaan Bisnis & Analisis (Bank Marketing)**

* **Masalah:** Mengidentifikasi karakteristik klien yang paling cenderung berlangganan deposito berjangka untuk meningkatkan efisiensi kampanye pemasaran.
* **Pengguna:** Tim Pemasaran Bank dan Analis Finansial.
* **Pertanyaan Analisis:**
  1. Bagaimana pengaruh tingkat pendidikan (`education`) dan status pernikahan (`marital`) terhadap keputusan berlangganan deposito (`y`)?
  2. Apakah klien dengan saldo tabungan rata-rata tahunan (`balance`) yang lebih tinggi lebih cenderung berlangganan?
  3. Bagaimana hubungan antara kelompok usia (`age`) klien dengan hasil kampanye pemasaran?

---
### **1. Instalasi Pustaka & Unduh Dataset Bank Marketing**

Menginstal library `ucimlrepo` dan `pyspark`, kemudian mengunduh dataset secara terprogram dari UCI ML Repository dan menyimpannya sebagai file CSV mentah.

In [ ]:
# Instal ucimlrepo dan pyspark
!pip install ucimlrepo pyspark -q

# Unduh dataset Bank Marketing dari UCI
from ucimlrepo import fetch_ucirepo
import pandas as pd

# Fetch dataset
bank_marketing = fetch_ucirepo(id=222)

# Ekstrak fitur dan target
X = bank_marketing.data.features
y = bank_marketing.data.targets

# Gabungkan menjadi satu DataFrame
full_df = pd.concat([X, y], axis=1)

# Simpan ke CSV untuk sumber pipeline
full_df.to_csv("bank_marketing.csv", index=False)
print("Dataset Bank Marketing berhasil disimpan sebagai 'bank_marketing.csv'")
print(f"Dimensi data mentah: {full_df.shape[0]} baris x {full_df.shape[1]} kolom")

---
### **2. Inisialisasi Sesi Spark & Memuat Data ke Spark DataFrame**

In [ ]:
from pyspark.sql import SparkSession

# Inisialisasi Spark Session
spark = SparkSession.builder \
    .appName("BankMarketing_01_EDA_Bronze") \
    .getOrCreate()

# Memuat dataset Bank Marketing ke Spark DataFrame
df_spark = spark.read.csv("bank_marketing.csv", header=True, inferSchema=True)

# Membuat Temporary View untuk Spark SQL
df_spark.createOrReplaceTempView("bank_table")

# Menampilkan 5 baris pertama
df_spark.show(5)

---
### **3. Eksplorasi Data Awal (Data Quality Check)**

Pemeriksaan kualitas data mencakup verifikasi skema tipe data, statistik deskriptif numerik, deteksi missing values, dan deteksi duplikasi baris.

In [ ]:
# A. Deskripsi Struktur & Statistik Data Bank Marketing
print("=== Skema Data Bank Marketing ===")
df_spark.printSchema()

print("\n=== Statistik Deskriptif ===")
df_spark.describe().show()

# B. Pengecekan Missing Value (Null)
from pyspark.sql.functions import col, sum as spark_sum

print("=== Jumlah Missing Value per Kolom ===")
df_spark.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df_spark.columns]).show()

# C. Pengecekan Duplikasi
total_baris = df_spark.count()
baris_unik = df_spark.distinct().count()
duplikasi = total_baris - baris_unik
print(f"Total baris: {total_baris}")
print(f"Baris unik: {baris_unik}")
print(f"Jumlah baris duplikat: {duplikasi}")

---
### **4. Analisis Agregasi Eksploratif (Spark SQL)**

Menjawab pertanyaan bisnis awal menggunakan query analitik Spark SQL.

In [ ]:
# Agregasi 1: Pengaruh Tingkat Pendidikan (education) & Pernikahan (marital) terhadap Keputusan Deposito (y)
print("=== Agregasi 1: Pendidikan & Status Pernikahan vs Keputusan Deposito ===")
spark.sql("""
    SELECT education, marital, y,
           COUNT(*) AS jumlah_klien,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(PARTITION BY education, marital), 2) AS persentase
    FROM bank_table
    GROUP BY education, marital, y
    ORDER BY education, marital, y
""").show()

# Agregasi 2: Rata-rata Saldo Tabungan Tahunan (balance) terhadap Keputusan Deposito (y)
print("\n=== Agregasi 2: Rata-rata Saldo (balance) vs Keputusan Deposito ===")
spark.sql("""
    SELECT y,
           ROUND(AVG(balance), 2) AS rata_rata_saldo,
           COUNT(*) AS jumlah_klien
    FROM bank_table
    GROUP BY y
""").show()

# Agregasi 3: Hubungan Kelompok Usia dengan Hasil Kampanye
print("\n=== Agregasi 3: Kelompok Usia vs Hasil Kampanye ===")
spark.sql("""
    SELECT CASE
                WHEN age < 30 THEN 'Muda (<30)'
                WHEN age BETWEEN 30 AND 50 THEN 'Paruh Baya (30-50)'
                ELSE 'Tua (>50)'
           END AS kelompok_usia,
           y,
           COUNT(*) AS jumlah_klien
    FROM bank_table
    GROUP BY kelompok_usia, y
    ORDER BY kelompok_usia, y
""").show()

---
### **5. Bronze Layer Ingestion (Raw Data Ingestion & Metadata Tagging)**

Pada layer **Bronze**, data disimpan dalam format aslinya (raw/unfiltered) dengan format penyimpanan teroptimasi (**Parquet**). 
Selain data asli, ditambahkan metadata audit:
* `ingested_at`: Waktu data di-ingest ke data lakehouse.
* `source_file`: Nama file sumber data untuk keperluan data lineage dan tracking.

In [ ]:
# Persiapan Direktori Penyimpanan Lakehouse
import os

os.makedirs("lakehouse/bronze", exist_ok=True)
os.makedirs("lakehouse/silver", exist_ok=True)
os.makedirs("lakehouse/gold", exist_ok=True)
os.makedirs("lakehouse/exports", exist_ok=True)
print("Direktori Lakehouse berhasil disiapkan.")

In [ ]:
from pyspark.sql.functions import current_timestamp, lit

# Ambil data mentah Bank Marketing
raw_df = spark.read.csv("bank_marketing.csv", header=True, inferSchema=True)

# Tambahkan metadata audit
bronze_df = raw_df \
    .withColumn("ingested_at", current_timestamp()) \
    .withColumn("source_file", lit("bank_marketing.csv"))

# Simpan ke penyimpanan Bronze (format Parquet)
bronze_path = "lakehouse/bronze/bank_marketing"
bronze_df.write.mode("overwrite").parquet(bronze_path)

print(f"Layer Bronze: Berhasil memuat dan menyimpan data mentah ke '{bronze_path}'")
print(f"Total baris tersimpan di Bronze: {bronze_df.count()}")

# Verifikasi membaca kembali dari layer Bronze
print("\n=== Sampel Data di Layer Bronze ===")
spark.read.parquet(bronze_path).show(5)

# Ekspor ke CSV Tunggal untuk integrasi eksternal
bronze_export_path = "lakehouse/exports/bronze_export"
bronze_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(bronze_export_path)
print(f"Ekspor Bronze ke CSV tunggal berhasil di '{bronze_export_path}'")

---
### **Ringkasan Layer Bronze**
* Data mentah berhasil tersimpan secara terstruktur di `lakehouse/bronze/bank_marketing` format Parquet.
* Metadata penelusuran (`ingested_at`, `source_file`) telah disematkan.
* Data siap dikonsumsi oleh **Layer Silver** pada notebook berikutnya (`02_Silver_Layer.ipynb`).